# BikeToDrive SDM vullen

Dit notebook voert stap 5 uit:

1. **reset** van alle tabellen in het Source Data Model  
2. **inladen** van data uit de 5 operationele SQLite-bronnen naar het SDM  
3. **controle** van aantallen en foreign keys

## Gekozen inlaadstrategie
Hier wordt een **full refresh / truncate-and-load** strategie gebruikt:

- eerst worden alle SDM-tabellen leeggemaakt
- daarna wordt elke bron **1-op-1** ingeladen in de bijbehorende SDM-tabellen
- de laadvolgorde volgt de foreign keys: eerst stamtabellen, daarna transactietabellen

Dat past hier goed, omdat:
- het SDM volledig opnieuw opgebouwd mag worden
- de brondata relatief klein is
- de SDM-tabellen bron-specifiek zijn benoemd, waardoor records uit verschillende databases elkaar niet overschrijven


In [5]:
from pathlib import Path
import sqlite3
import pandas as pd
import logging

BASE_DIR = Path.home() / "Downloads"

SOURCE_DBS = {
    "accessoireverkoop": BASE_DIR / "BikeToDrive_1_Accessoireverkoop.db",
    "fietsverkoop": BASE_DIR / "BikeToDrive_2_Fietsverkoop.db",
    "onderhoud": BASE_DIR / "BikeToDrive_3_Onderhoud.db",
    "accessoire_inkoop": BASE_DIR / "BikeToDrive_4_Accessoire_Inkoop.db",
    "fiets_inkoop": BASE_DIR / "BikeToDrive_5_Fiets_Inkoop.db",
}

SDM_DB = BASE_DIR / "BikeToDrive_SDM.db"

TABLE_MAPPING = {
    "accessoireverkoop": [
        ("Filiaal", "Accessoire_Verkoop_Filiaal"),
        ("Klant", "Accessoire_Verkoop_Klant"),
        ("Leverancier", "Accessoire_Verkoop_Leverancier"),
        ("Monteur", "Accessoire_Verkoop_Monteur"),
        ("Accessoire", "Accessoire_Verkoop_Accessoire"),
        ("Accessoire_Verkoop", "Accessoire_Verkoop"),
    ],
    "fietsverkoop": [
        ("Filiaal", "Fiets_Verkoop_Filiaal"),
        ("Klant", "Fiets_Verkoop_Klant"),
        ("Fabrikant", "Fiets_Verkoop_Fabrikant"),
        ("Monteur", "Fiets_Verkoop_Monteur"),
        ("Fiets", "Fiets_Verkoop_Fiets"),
        ("Fiets_Verkoop", "Fiets_Verkoop"),
    ],
    "onderhoud": [
        ("Filiaal", "Onderhoud_Filiaal"),
        ("Fabrikant", "Onderhoud_Fabrikant"),
        ("Fiets", "Onderhoud_Fiets"),
        ("Monteur", "Onderhoud_Monteur"),
        ("Onderhoud", "Onderhoud"),
    ],
    "accessoire_inkoop": [
        ("Leverancier", "Accessoire_Inkoop_Leverancier"),
        ("Accessoire", "Accessoire_Inkoop_Accessoire"),
        ("Accessoire_Inkoop", "Accessoire_Inkoop"),
    ],
    "fiets_inkoop": [
        ("Fabrikant", "Fiets_Inkoop_Fabrikant"),
        ("Fiets", "Fiets_Inkoop_Fiets"),
        ("Fiets_Inkoop", "Fiets_Inkoop"),
    ],
}

# -----------------------------
# LOGGING CONFIGURATIE
# -----------------------------
LOG_DIR = BASE_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

SDM_LOG_PATH = LOG_DIR / "sdm_etl.log"

logger = logging.getLogger("sdm_etl")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

file_handler = logging.FileHandler(SDM_LOG_PATH, encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s|%(levelname)s|%(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.propagate = False

def log_event(level, process_step, table_name="", action="", row_count="", details=""):
    message = f"{process_step}|{table_name}|{action}|{row_count}|{details}"
    if level == "INFO":
        logger.info(message)
    elif level == "WARNING":
        logger.warning(message)
    elif level == "ERROR":
        logger.error(message)
    else:
        logger.info(message)

# -----------------------------
# BESTANDSCONTROLE
# -----------------------------
for label, path in SOURCE_DBS.items():
    if not path.exists():
        log_event("ERROR", "FILE_CHECK", label, "MISSING_SOURCE_DB", "", str(path))
        raise FileNotFoundError(f"Bronbestand ontbreekt: {path}")
    else:
        log_event("INFO", "FILE_CHECK", label, "FOUND_SOURCE_DB", "", str(path))

if not SDM_DB.exists():
    log_event("ERROR", "FILE_CHECK", "SDM", "MISSING_SDM_DB", "", str(SDM_DB))
    raise FileNotFoundError(f"SDM-bestand ontbreekt: {SDM_DB}")
else:
    log_event("INFO", "FILE_CHECK", "SDM", "FOUND_SDM_DB", "", str(SDM_DB))

print("Bestanden gevonden.")
print("SDM:", SDM_DB)
for label, path in SOURCE_DBS.items():
    print(f"- {label}: {path.name}")

def reset_sdm_log():
    if SDM_LOG_PATH.exists():
        SDM_LOG_PATH.unlink()
    open(SDM_LOG_PATH, "w", encoding="utf-8").close()
    print(f"SDM-log gereset: {SDM_LOG_PATH}")

    global logger
    if logger.hasHandlers():
        logger.handlers.clear()

    file_handler = logging.FileHandler(SDM_LOG_PATH, encoding="utf-8")
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.propagate = False

def get_user_tables(conn: sqlite3.Connection) -> list[str]:
    rows = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """).fetchall()
    return [row[0] for row in rows]

def reset_sdm(sdm_path: Path) -> None:
    log_event("INFO", "RESET_SDM", "", "START", "", f"SDM reset gestart voor {sdm_path.name}")

    with sqlite3.connect(sdm_path) as conn:
        conn.execute("PRAGMA foreign_keys = OFF")
        tables = get_user_tables(conn)

        total_deleted_tables = 0
        for table in tables:
            before_count = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
            conn.execute(f'DELETE FROM "{table}"')
            log_event("INFO", "RESET_SDM", table, "DELETE_ROWS", before_count, "Tabel leeggemaakt")
            total_deleted_tables += 1

        conn.commit()
        conn.execute("PRAGMA foreign_keys = ON")

    log_event("INFO", "RESET_SDM", "", "END", total_deleted_tables, "Aantal tabellen leeggemaakt")
    print(f"{len(tables)} tabellen leeggemaakt.")

def fetch_all_rows(conn: sqlite3.Connection, table_name: str):
    conn.row_factory = sqlite3.Row
    rows = conn.execute(f'SELECT * FROM "{table_name}"').fetchall()
    log_event("INFO", "EXTRACT", table_name, "READ_SOURCE_TABLE", len(rows), "Brondata gelezen")
    return rows

def load_table(source_conn: sqlite3.Connection, target_conn: sqlite3.Connection, source_table: str, target_table: str) -> int:
    rows = fetch_all_rows(source_conn, source_table)

    if not rows:
        log_event("INFO", "LOAD_TO_SDM", target_table, "INSERT", 0, f"Geen rijen in bron_tabel {source_table}")
        return 0

    columns = list(rows[0].keys())
    column_sql = ", ".join([f'"{col}"' for col in columns])
    placeholders = ", ".join(["?"] * len(columns))

    target_conn.executemany(
        f'INSERT INTO "{target_table}" ({column_sql}) VALUES ({placeholders})',
        [tuple(row[col] for col in columns) for row in rows]
    )

    log_event(
        "INFO",
        "LOAD_TO_SDM",
        target_table,
        "INSERT",
        len(rows),
        f"Bron_tabel={source_table}"
    )
    return len(rows)

def load_all_sources(sdm_path: Path, source_dbs: dict, table_mapping: dict) -> pd.DataFrame:
    log_event("INFO", "LOAD_ALL_SOURCES", "", "START", "", "Laden van bronnen naar SDM gestart")
    results = []

    with sqlite3.connect(sdm_path) as target_conn:
        target_conn.execute("PRAGMA foreign_keys = OFF")

        for source_label, mappings in table_mapping.items():
            log_event("INFO", "SOURCE_LOOP", source_label, "OPEN_SOURCE_DB", "", str(source_dbs[source_label]))

            with sqlite3.connect(source_dbs[source_label]) as source_conn:
                for source_table, target_table in mappings:
                    try:
                        loaded_rows = load_table(source_conn, target_conn, source_table, target_table)
                        results.append({
                            "bron": source_label,
                            "bron_tabel": source_table,
                            "sdm_tabel": target_table,
                            "geladen_rijen": loaded_rows
                        })
                    except Exception as e:
                        log_event(
                            "ERROR",
                            "LOAD_ALL_SOURCES",
                            target_table,
                            "FAILED",
                            "",
                            f"Bron={source_label}; bron_tabel={source_table}; fout={str(e)}"
                        )
                        raise

        target_conn.commit()
        target_conn.execute("PRAGMA foreign_keys = ON")

    total_rows = sum(r["geladen_rijen"] for r in results)
    log_event("INFO", "LOAD_ALL_SOURCES", "", "END", total_rows, "Alle bronnen geladen naar SDM")

    return pd.DataFrame(results)

def foreign_key_issues(sdm_path: Path):
    with sqlite3.connect(sdm_path) as conn:
        issues = conn.execute("PRAGMA foreign_key_check").fetchall()

    if issues:
        log_event("WARNING", "FOREIGN_KEY_CHECK", "", "ISSUES_FOUND", len(issues), str(issues))
    else:
        log_event("INFO", "FOREIGN_KEY_CHECK", "", "NO_ISSUES", 0, "Geen foreign key problemen gevonden")

    return issues

def row_counts(sdm_path: Path) -> pd.DataFrame:
    with sqlite3.connect(sdm_path) as conn:
        tables = get_user_tables(conn)
        data = []

        for table in tables:
            count = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
            data.append({"tabel": table, "aantal_rijen": count})
            log_event("INFO", "ROW_COUNT", table, "COUNT_ROWS", count, "Rijtelling uitgevoerd")

    return pd.DataFrame(data).sort_values(["tabel"]).reset_index(drop=True)

def run_sdm_pipeline():
    log_event("INFO", "RUN_SDM_PIPELINE", "", "START", "", "SDM pipeline gestart")

    reset_sdm(SDM_DB)
    load_result = load_all_sources(SDM_DB, SOURCE_DBS, TABLE_MAPPING)
    issues = foreign_key_issues(SDM_DB)
    counts = row_counts(SDM_DB)

    log_event("INFO", "RUN_SDM_PIPELINE", "", "END", "", "SDM pipeline succesvol afgerond")

    return load_result, issues, counts

Bestanden gevonden.
SDM: /Users/cmok/Downloads/BikeToDrive_SDM.db
- accessoireverkoop: BikeToDrive_1_Accessoireverkoop.db
- fietsverkoop: BikeToDrive_2_Fietsverkoop.db
- onderhoud: BikeToDrive_3_Onderhoud.db
- accessoire_inkoop: BikeToDrive_4_Accessoire_Inkoop.db
- fiets_inkoop: BikeToDrive_5_Fiets_Inkoop.db


## 1. Reset-knop: alle SDM-tabellen leegmaken

In [6]:
reset_sdm(SDM_DB)
reset_sdm_log()

23 tabellen leeggemaakt.
SDM-log gereset: /Users/cmok/Downloads/logs/sdm_etl.log


## 2. Data uit alle 5 bronbestanden overzetten naar het SDM

De laadvolgorde is per bron zo gekozen dat eerst de tabellen zonder afhankelijke foreign keys worden geladen en daarna de transactietabellen.


In [7]:
load_result = load_all_sources(SDM_DB, SOURCE_DBS, TABLE_MAPPING)
load_result

,bron,bron_tabel,sdm_tabel,geladen_rijen
0,accessoireverkoop,Filiaal,Accessoire_Verkoop_Filiaal,4
1,accessoireverkoop,Klant,Accessoire_Verkoop_Klant,20
2,accessoireverkoop,Leverancier,Accessoire_Verkoop_Leverancier,5
3,accessoireverkoop,Monteur,Accessoire_Verkoop_Monteur,10
4,accessoireverkoop,Accessoire,Accessoire_Verkoop_Accessoire,10
5,accessoireverkoop,Accessoire_Verkoop,Accessoire_Verkoop,100
6,fietsverkoop,Filiaal,Fiets_Verkoop_Filiaal,4
7,fietsverkoop,Klant,Fiets_Verkoop_Klant,25
8,fietsverkoop,Fabrikant,Fiets_Verkoop_Fabrikant,10
9,fietsverkoop,Monteur,Fiets_Verkoop_Monteur,10


## 3. Controle van de inhoud

In [4]:
counts_df = row_counts(SDM_DB)
counts_df

,tabel,aantal_rijen
0,Accessoire_Inkoop,50
1,Accessoire_Inkoop_Accessoire,13
2,Accessoire_Inkoop_Leverancier,5
3,Accessoire_Verkoop,100
4,Accessoire_Verkoop_Accessoire,10
5,Accessoire_Verkoop_Filiaal,4
6,Accessoire_Verkoop_Klant,20
7,Accessoire_Verkoop_Leverancier,5
8,Accessoire_Verkoop_Monteur,10
9,Fiets_Inkoop,100


## 4. Foreign key controle

In [5]:
fk_errors = foreign_key_issues(SDM_DB)
print('Aantal foreign key issues:', len(fk_errors))
fk_errors[:10]

Aantal foreign key issues: 0


[]

## Conclusie

Als `Aantal foreign key issues: 0` wordt getoond, dan is het SDM correct gevuld en zijn de verwijzingen geldig.
